## Agent Traces to Supervised FineTuning

This demo shows how Agent Traces from a tool-calling Hosted Agent can be used to perform Supervised FineTuning (SFT) on an AzureOpenAI model. 

In [4]:
%pip install azure_ai_projects-2.2.0-py3-none-any.whl

In [1]:
# TODO: add command to grant Logs Reader role

In [1]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential
)

In [2]:
page = project_client.beta.datasets.list_generation_jobs()
jobs = list(page)
print(jobs[0].id, jobs[0].inputs.name)

datagen-f6ec59c70e3649c287d5399548464812 sft_traces_data


In [4]:
job = project_client.beta.datasets.get_generation_job(jobs[0].id)
print(job.id, job.inputs.name)

datagen-f6ec59c70e3649c287d5399548464812 sft_traces_data


In [5]:
from azure.ai.projects.models import (
    DataGenerationJob,
    DataGenerationJobInputs,
    DataGenerationJobScenario,
    TracesDataGenerationJobOptions,
    TracesDataGenerationJobSource,
)

# HACK: reorder _data dict so "type" is serialized first — the service rejects the payload otherwise.
def _put_type_first(model):
    if hasattr(model, "_data") and "type" in model._data:
        model._data = {"type": model._data["type"], **{k: v for k, v in model._data.items() if k != "type"}}

options = TracesDataGenerationJobOptions(
    max_samples=50,
    train_split=0.8
)
_put_type_first(options)

source = TracesDataGenerationJobSource(
    agent_name="aprilk-tracebed-af-responses",
    start_time=1778070134
)
_put_type_first(source)

job = DataGenerationJob(
    inputs=DataGenerationJobInputs(
        name="sft_traces_data",
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        options=options,
        sources=[source]
    )
)

job = project_client.beta.datasets.create_generation_job(job)

In [6]:
job = project_client.beta.datasets.get_generation_job(job.id)
print(job.status)

JobStatus.IN_PROGRESS
